# Job Category Classifier

Predict which of six job-board categories a job ad belongs to, from its title and text.

## 1. Load the data

The pinned dataset file is downloaded into `data/raw/` (see the README).

In [3]:
import numpy as np
import pandas as pd

df = pd.read_parquet('data/raw/djinni-jobs-en.parquet')
df.shape

(141897, 10)

## 2. First look

Columns, example rows and data types.

In [4]:
df.head().T

,0,1,2,3,4
Position,10 + Blockchain Nodes / Masternodes to set up,10 .NET Developers (Middle and Senior level),"10X Engineer (co-founder, #4 employee, USD 11-...",16 - Amazon Brand Manager,16 - Amazon Brand Manager
Long Description,*Requirements*\r\n\r\nWe're looking for a long...,"Greetings! My name is Maria, I am in urgent ne...",**Product**\r\nThe product is a live video cha...,"Currently, TCM expanding its activities to Ukr...","Hello,\r\nWe, MIMIRB2B, are an outstaff compan..."
Company Name,MyCointainer,TechScout.tech,Innoteka,FirstFive,MimirB2B
Exp Years,2y,2y,5y,2y,1y
Primary Keyword,Sysadmin,.NET,JavaScript,Marketing,Marketing
English Level,intermediate,intermediate,fluent,upper,upper
Published,2020-10-01T00:00:00+03:00,2022-03-01T00:00:00+02:00,2021-07-01T00:00:00+03:00,2022-01-01T00:00:00+02:00,2021-12-01T00:00:00+02:00
Long Description_lang,en,en,en,en,en
id,c0ca96e7-85df-50df-a64e-d934cd02a170,64f4b7ea-36e4-5bdd-a8b1-185f32f7dc7f,b9a1303e-dd0c-5ed1-8f62-be2bc4c7da4f,99cb3f4a-9b4b-53d9-9a3b-bab2c22da346,bc1419f7-28e2-582b-8d53-22e28b2f0210
__index_level_0__,27461,27462,27463,27464,27465


In [5]:
df.dtypes

Position                   str
Long Description           str
Company Name               str
Exp Years                  str
Primary Keyword            str
English Level              str
Published                  str
Long Description_lang      str
id                         str
__index_level_0__        int64
dtype: object

Column names in lower case with `_` instead of spaces, as in the course.

In [8]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.columns

Index(['position', 'long_description', 'company_name', 'exp_years',
       'primary_keyword', 'english_level', 'published',
       'long_description_lang', 'id', '__index_level_0__'],
      dtype='str')

## 3. Missing values

Only `english_level` has missing values. It is a categorical column, so missing values become `'NA'`.
Then we look for hidden missing values: empty strings or strings of spaces, which pandas does not count as missing.

In [9]:
df.isnull().sum()

position                    0
long_description            0
company_name                0
exp_years                   0
primary_keyword             0
english_level            7539
published                   0
long_description_lang       0
id                          0
__index_level_0__           0
dtype: int64

In [10]:
df.english_level = df.english_level.fillna('NA')
df.english_level.value_counts()

english_level
upper           66948
intermediate    54965
fluent          10583
NA               7539
pre              1689
basic             173
Name: count, dtype: int64

In [12]:
string_columns = list(df.dtypes[df.dtypes == 'str'].index)

In [13]:
for c in string_columns:
    s = df[c].str.strip()
    print(c, 'empty:', (s == '').sum(), 'shortest:', s.str.len().min())

position empty: 0 shortest: 2
long_description empty: 0 shortest: 51
company_name empty: 0 shortest: 2
exp_years empty: 0 shortest: 2
primary_keyword empty: 0 shortest: 2
english_level empty: 0 shortest: 2
published empty: 0 shortest: 25
long_description_lang empty: 0 shortest: 2
id empty: 0 shortest: 36


## 4. Publication date

`published` holds midnight on the first day of a month in Ukrainian time, for example `2020-10-01T00:00:00+03:00`.
Converting it to UTC moves every ad to the previous month (shown below), so only the date part is kept.

In [14]:
pd.to_datetime(df.published.head(3), utc=True)

0   2020-09-30 21:00:00+00:00
1   2022-02-28 22:00:00+00:00
2   2021-06-30 21:00:00+00:00
Name: published, dtype: datetime64[us, UTC]

In [15]:
df.published = pd.to_datetime(df.published.str[:10])
df.published.min(), df.published.max(), df.published.nunique()

(Timestamp('2020-01-01 00:00:00'), Timestamp('2023-09-01 00:00:00'), 45)

## 5. Duplicates

The same ad in both training and test data would make the model look better than it is.
We check repeated ids, identical rows, identical texts after normalizing case and spaces, and texts whose first 300 characters appear under more than one company.

In [16]:
df.id.duplicated().sum(), df.drop(columns='__index_level_0__').duplicated().sum()

(np.int64(0), np.int64(0))

In [17]:
description_norm = df.long_description.str.lower().str.split().str.join(' ')
description_norm.duplicated().sum()

np.int64(0)

In [18]:
df['description_start'] = description_norm.str[:300]
df['company_norm'] = df.company_name.str.lower().str.strip()

In [19]:
companies_per_start = df.groupby('description_start').company_norm.nunique()
shared = companies_per_start[companies_per_start > 1]
len(shared), df.description_start.isin(shared.index).sum()

(329, np.int64(1010))

Examples of texts that start the same way under different company names. They are not duplicated ads: mostly the same company spelled differently (`scalors` / `scalors gmbh`, `intellectsoft` / `intlectsoft`, `clario` / `сlario` with a Cyrillic `с`) or an agency posting ads of other companies.

In [20]:
examples = df[df.description_start.isin(shared.index)].sort_values('description_start')
examples[['company_norm', 'position', 'primary_keyword', 'description_start']].head(15)

,company_norm,position,primary_keyword,description_start
3337,scalors,Android Software Developer,Android,**about client:** kivu technologies is an inno...
32590,scalors gmbh,Front-End Developer for Kivu,JavaScript,**about client:** kivu technologies is an inno...
59927,intellectsoft,Lead Full-Stack .NET,.NET,**about intellectsoft:** we are a digital tran...
62100,intellectsoft,Lead/Senior Java developer,Java,**about intellectsoft:** we are a digital tran...
124654,intellectsoft,Senior React Developer,JavaScript,**about intellectsoft:** we are a digital tran...
66605,intlectsoft,Middle Android Developer,Android,**about intellectsoft:** we are a digital tran...
62033,clario,Lead/Senior Affiliate Marketing Manager,Marketing,"**about our products** our product, with its i..."
134352,сlario,Talent Acquisition Specialist,HR,"**about our products** our product, with its i..."
40858,clario,Golang Developer (Mid or Senior),Golang,"**about our products** our product, with its i..."
109290,clario,Senior Data Engineer,Data Engineer,"**about our products** our product, with its i..."


## 6. Company groups

Splitting by company name alone could put the same company into both training and test data. Companies whose ads share the same first 300 characters form one group. The rule is applied until nothing changes, because company A can be linked to B and B to C. The data is later split by `company_group`.

The groups are built on all 45 categories, so links found through ads outside the six selected categories also count.

In [21]:
df['company_group'] = df.company_norm

while True:
    before = df.company_group.copy()
    df['company_group'] = df.groupby('description_start').company_group.transform('min')
    df['company_group'] = df.groupby('company_norm').company_group.transform('min')
    if (df.company_group == before).all():
        break

df.company_norm.nunique(), df.company_group.nunique()

(10197, 9924)

In [24]:
df.company_group.value_counts().head(10)

company_group
at it-rex - staffing services    9046
adaptiq                          3445
101ants                          3306
epam                             2032
a-hr                             1721
daxx                             1447
8allocate                        1443
benefit agency                   1231
chi                              1204
artkai                           1087
Name: count, dtype: int64

## 7. The six categories

The project keeps the six largest technical categories. The classes are imbalanced: Data Science is almost eight times smaller than JavaScript, which is why macro F1 is the main metric rather than accuracy.

In [25]:
categories = ['JavaScript', 'Java', 'DevOps', 'QA Automation', 'Python', 'Data Science']
df = df[df.primary_keyword.isin(categories)].reset_index(drop=True)
df.shape

(49638, 13)

In [26]:
df.primary_keyword.value_counts()

primary_keyword
JavaScript       17903
Java              8712
DevOps            7979
QA Automation     7047
Python            5735
Data Science      2262
Name: count, dtype: int64

In [27]:
df.primary_keyword.value_counts(normalize=True).round(3)

primary_keyword
JavaScript       0.361
Java             0.176
DevOps           0.161
QA Automation    0.142
Python           0.116
Data Science     0.046
Name: proportion, dtype: float64

## 8. Text for the model

The model reads the title and the ad text together. Line breaks and repeated spaces are collapsed into single spaces. Punctuation stays, because names such as `C++`, `C#` and `.NET` carry meaning.

In [28]:
df['text'] = (df.position + ' ' + df.long_description).str.split().str.join(' ')
df.text.iloc[0][:300]

'10X Engineer (co-founder, #4 employee, USD 11-12k, 1-4% equity) **Product** The product is a live video chat and co-browsing with website visitors. It helps companies generate more customers with instant video meetings directly on visitor’s website while they’re browsing a company’s page. It’s like '

In [29]:
df.text.str.len().describe()

count    49638.000000
mean      1745.891394
std        949.370331
min         77.000000
25%       1066.000000
50%       1576.000000
75%       2227.000000
max      12400.000000
Name: text, dtype: float64

## 9. Train, validation and test split

As in the course: 20% for test first, then 25% of the rest for validation, with `random_state=1`. The split is made over company groups, not rows, so every group lands in exactly one set.

Because some groups are large (the biggest holds 3,903 ads), the sets come out at 34,039 / 8,021 / 7,578 ads, about 69 / 16 / 15 instead of 60 / 20 / 20. Every category is present in every set. The seed is not changed to get nicer numbers: that would be choosing the split by its result.

In [30]:
from sklearn.model_selection import train_test_split

groups = df.company_group.drop_duplicates()
groups_full_train, groups_test = train_test_split(groups, test_size=0.2, random_state=1)
groups_train, groups_val = train_test_split(groups_full_train, test_size=0.25, random_state=1)

df_train = df[df.company_group.isin(groups_train)].reset_index(drop=True)
df_val = df[df.company_group.isin(groups_val)].reset_index(drop=True)
df_test = df[df.company_group.isin(groups_test)].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(34039, 8021, 7578)

In [31]:
(set(df_train.company_group) & set(df_val.company_group),
 set(df_train.company_group) & set(df_test.company_group),
 set(df_val.company_group) & set(df_test.company_group))

(set(), set(), set())

In [32]:
pd.DataFrame({
    'train': df_train.primary_keyword.value_counts(normalize=True),
    'val': df_val.primary_keyword.value_counts(normalize=True),
    'test': df_test.primary_keyword.value_counts(normalize=True),
}).round(3)

,train,val,test
primary_keyword,,,
Data Science,0.045,0.050,0.044
DevOps,0.164,0.146,0.160
Java,0.184,0.159,0.153
JavaScript,0.345,0.385,0.404
Python,0.111,0.134,0.117
QA Automation,0.150,0.126,0.122
